In [1]:
from dotenv import load_dotenv
from pathlib import Path
from IPython.display import Markdown, display
from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter
from langchain_google_genai import GoogleGenerativeAI, ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_chroma import Chroma
import gradio as gr
import os

load_dotenv()
# GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY_NEW")
GOOGLE_GEMINI_MODEL2 = os.getenv("GOOGLE_GEMINI_MODEL2")
model_name = "sentence-transformers/all-MiniLM-L6-v2"
DB_NAME = "vector_db"

c:\Users\MANAV\OneDrive\Desktop\Manav Code\code\Gen AI\python\GenAndAgenticAI\ai_with_qdrant\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
embedding = HuggingFaceEmbeddings(
    model_name = model_name
)
vector_store = Chroma(persist_directory=DB_NAME, embedding_function=embedding)

In [3]:
retriever = vector_store.as_retriever()
llm = ChatGoogleGenerativeAI(
    model = GOOGLE_GEMINI_MODEL2,
)

In [4]:
def get_all_user_data(dir_name : str = "knowledge-base"):
    content = ""
    dir_path = Path(dir_name)
    for file in dir_path.rglob("*"):
        if file.is_file():
            content += file.read_text(encoding="utf-8")
            content += "\n\n"
    return content

In [5]:
from langchain_core.documents.base import Document
headers_to_spilt_on = [
    ("#", "title"),
    ("##", "Subtitle"),
    ("###", "Section"),
]
def split_markdown_text(content : str) -> Document:
    markdown_splitter = MarkdownHeaderTextSplitter(
        headers_to_spilt_on,
    )
    return markdown_splitter.split_text(content)

def split_docs_using_recursive_text_splitter(documents : list[Document]):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 800,
        chunk_overlap = 200,
    )
    return text_splitter.split_documents(documents)

In [6]:
def split_company_content(content : str):
    markdown_docs = split_markdown_text(content)
    all_chunks = []
    for doc in markdown_docs:
        small_chunk = split_docs_using_recursive_text_splitter([doc])
        all_chunks.extend(small_chunk)
    return all_chunks

In [7]:
docs = split_company_content(get_all_user_data())
len(docs)
docs[0]

Document(metadata={'title': 'About Insurellm'}, page_content='Insurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.  \nThe company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, Insurellm had reached a peak of 200 employees with 12 offices across the US.')

In [22]:
vector_store.add_documents(documents=docs)

['b874a6d8-3a3b-4af3-be89-dc487c0b71d9',
 '2cd3f744-35ce-411d-9803-376f7ded2f5f',
 '15bd88f1-6dcd-4da7-bab6-bc2a10106112',
 'cf84ee11-ac4f-4b1f-bf77-50ea49afecca',
 '4a971c7a-a2e2-455f-b80a-2cbed01a271f',
 '13c50dae-97b5-4078-95c3-b4a4b2f25f00',
 '8fefb8c4-5acc-4ebc-bc02-0105b18c9e1b',
 '262902ea-dd98-401e-b339-8f644762f61f',
 '4817744b-9a26-4485-b7a2-5db7c73e0782',
 '37782dee-6970-44b9-8cf2-e3c90a3174cb',
 '20255ace-c144-48d7-b644-83d8f4ed7ce2',
 '304c1d8c-8b0b-4c55-b354-27d910fea31c',
 'e7dae049-5441-4741-8f55-0215dac02ad5',
 'a72224d0-4ad7-432b-abec-8e51a6791526',
 'd7575f44-edcf-4f28-8163-3a8a72fc8cca',
 '826bdb1c-665a-4504-98e3-ef2f15455850',
 'af563de6-21aa-44bd-9828-47acb801922b',
 '4a23f1a8-97c6-42f7-b279-04fa97853e25',
 'a64ecf7c-f229-4530-858d-16ee1bff4f98',
 '7e9ce0c7-01dd-47ea-96f7-9934590f3cc5',
 'f6aa9130-8f87-49f3-a3ed-0195157f9acc',
 'd13bee50-bc51-4bc0-9a53-56e2f982fafb',
 '0eb6abf0-7a51-4d8a-99a2-ea02143fc4bc',
 '818c8fdb-9b89-4442-b0f3-f78145c23a25',
 '0c46e5b1-2dc8-

In [14]:
retriever.invoke("Who is Avery?")

[Document(id='38b1cc31-8fff-4324-afeb-6d4accf8b059', metadata={'Subtitle': 'Insurellm Career Progression', 'title': 'Avery Lancaster'}, page_content='- **2010 - 2013**: Business Analyst at Edge Analytics\nPrior to joining Innovate, Avery worked as a Business Analyst, focusing on market trends and consumer preferences in the insurance space. This position laid the groundwork for Avery’s future entrepreneurial endeavors.'),
 Document(id='9bf914e7-4036-4709-bf20-9437c0123d98', metadata={'Subtitle': 'Other HR Notes', 'title': 'Avery Lancaster'}, page_content="- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.\n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.\n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Aver

In [ ]:
llm.invoke("Who is Avery?")

In [11]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [9]:
def question_answer(user_query, history = []):
    relevent_docs = retriever.invoke(user_query)
    context = "\n\n".join(doc.page_content for doc in relevent_docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(system_prompt), HumanMessage(user_query)])
    return response.content

In [ ]:
ans = question_answer("Who is Averi Lancaster?", [])



[SystemMessage(content="\nYou are a knowledgeable, friendly assistant representing the company Insurellm.\nYou are chatting with a user about Insurellm.\nIf relevant, use the given context to answer any question.\nIf you don't know the answer, say so.\nContext:\nAvery Lancaster has demonstrated resilience and adaptability throughout her career at Insurellm, positioning the company as a key player in the insurance technology landscape.\n\n- **2015 - Present**: Co-Founder & CEO\nAvery Lancaster co-founded Insurellm in 2015 and has since guided the company to its current position as a leading Insurance Tech provider. Avery is known for her innovative leadership strategies and risk management expertise that have catapulted the company into the mainstream insurance market.  \n- **2013 - 2015**: Senior Product Manager at Innovate Insurance Solutions\nBefore launching Insurellm, Avery was a leading Senior Product Manager at Innovate Insurance Solutions, where she developed groundbreaking insu

In [31]:
display(Markdown(ans))

Avery Lancaster is the Co-Founder and CEO of Insurellm, a position she has held since co-founding the company in 2015. She is recognized for her innovative leadership and risk management expertise, which have been instrumental in positioning Insurellm as a leading Insurance Tech provider.

Before Insurellm, she was a Senior Product Manager at Innovate Insurance Solutions and a Business Analyst at Edge Analytics. Avery is also involved in mentorship and the Women in Tech initiative at Insurellm.

In [12]:
gr.ChatInterface(question_answer).launch()

c:\Users\MANAV\OneDrive\Desktop\Manav Code\code\Gen AI\python\GenAndAgenticAI\ai_with_qdrant\.venv\Lib\site-packages\gradio\chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [82]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

In [93]:
str_parser = StrOutputParser()
system_prompt = PromptTemplate(
    template=SYSTEM_PROMPT_TEMPLATE,
    input_variables=["context"],
    validate_template=True,
)

system_msg_pipline = (
    retriever 
    | RunnableLambda(lambda documents : "\n\n".join(doc.page_content for doc in documents)) 
    | system_prompt
    ) 
system_msg_pipline.invoke("Who is Avery?")

StringPromptValue(text="\nYou are a knowledgeable, friendly assistant representing the company Insurellm.\nYou are chatting with a user about Insurellm.\nIf relevant, use the given context to answer any question.\nIf you don't know the answer, say so.\nContext:\n- **2010 - 2013**: Business Analyst at Edge Analytics\nPrior to joining Innovate, Avery worked as a Business Analyst, focusing on market trends and consumer preferences in the insurance space. This position laid the groundwork for Avery’s future entrepreneurial endeavors.\n\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.\n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.\n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by imp

In [94]:
chat_template = ChatPromptTemplate([
    ("system", "{system_msg}"),
    ("human", "{user_query}"),
])
pipline = (RunnableParallel({
    "system_msg" : system_msg_pipline,
    "user_query" : RunnableLambda(lambda x : x)
    }) 
    | chat_template 
    | llm 
    | str_parser
)
res = pipline.invoke("What is InsureLLm?")


In [95]:
display(Markdown(res))

Insurellm is an innovative insurance tech firm founded in 2015 by Avery Lancaster. It started as a high-growth startup and has evolved into a lean, profitable operation focused on sustainable growth and operational excellence.

The company offers a portfolio of products designed to enhance the insurance experience, including:
*   **Markellm:** A marketplace connecting consumers with insurance providers.
*   **Carllm:** An auto insurance portal.
*   **Homellm:** A home insurance portal.
*   **Rellm:** An enterprise reinsurance platform.

Insurellm operates primarily remotely across the US with a team of 32 employees, maintaining its headquarters in San Francisco and satellite offices in New York, Austin, Chicago, and Denver.

In [99]:
str_parser = StrOutputParser()
system_prompt_template = PromptTemplate(
    template=SYSTEM_PROMPT_TEMPLATE,
    input_variables=["context"],
    validate_template=True,
)
chat_template = ChatPromptTemplate([
        ("system", "{system_msg}"),
        ("human", "{user_query}"),
])
def question_answer2(user_query : str, history):
    system_msg_pipline = (
        retriever 
        | RunnableLambda(lambda documents : "\n\n".join(doc.page_content for doc in documents)) 
        | system_prompt_template
        ) 
    final_prompt_pipline = (
        RunnableParallel({
        "system_msg" : system_msg_pipline,
        "user_query" : RunnableLambda(lambda x : x)
        }) 
        | chat_template
    )
    pipline =  ( 
        final_prompt_pipline
        | llm 
        | str_parser
    )
    return pipline.invoke(user_query)

In [ ]:
gr.ChatInterface(question_answer).launch()
